Python functions for analyzing SNOTEL data

In [2]:
import numpy as np
from numba import njit

# Loading and processing data

In [16]:
def csvToXarray(dataPath,for2526=False):
    '''Load and convert SNOTEL data from a CSV to an xarray Dataset.
    Inputs:
        - stationDataPath: string specifying location of station data
        - for2526: boolean, set equal to true if you are loading data from water year 2025-2026 (this was downloaded 4/22 and is thus not complete)
    Outputs:
        - stationXarray:
            xarray Dataset with a water year coordinate. Has variables T_avg, swe, and prcp
    '''
    
    # load in station csv
    stationDataFrame = pd.read_csv(dataPath, delimiter=',')
    
    # extract variables and convert units
    time = stationDataFrame.date
    time_dt64 = time.values.astype('datetime64[D]')
    swe_in = stationDataFrame.swe_in
    swe_mm = swe_in*25.4
    prcp_in = stationDataFrame.precipIncrem_in
    prcp_mm = prcp_in*25.4
    prcpTot_in = stationDataFrame.precipAccum_in
    prcpTot_mm = prcpTot_in*25.4
    temp_F = stationDataFrame.Tavg_F
    temp_C = (temp_F-32)*(5/9)
    
    # put into xarray Dataset
    stationXarray = xr.Dataset()
    tempArray = xr.DataArray(temp_C.values,
                             coords={'time':time_dt64},
                             dims='time').assign_attrs(units="Celsius",description="daily mean temperature")
    tempArray[tempArray==-np.inf] = np.nan
    
    sweArray = xr.DataArray(swe_mm.values,
                            coords={'time':time_dt64},
                            dims='time').assign_attrs(units="mm",description="daily snow water equivalent")
    sweArray[sweArray==-np.inf] = np.nan
    
    precipArray = xr.DataArray(prcp_mm.values,
                               coords={'time':time_dt64},
                               dims='time').assign_attrs(units="mm",description="daily precipitation")
    precipArray[precipArray==-np.inf] = np.nan
    
    precipTotArray = xr.DataArray(prcpTot_mm.values,
                               coords={'time':time_dt64},
                               dims='time').assign_attrs(units="mm",description="daily precipitation")
    precipTotArray[precipTotArray==-np.inf] = np.nan

    stationXarray['T_avg'] = tempArray
    stationXarray['swe'] = sweArray
    stationXarray['prcp'] = precipArray
    stationXarray['prcpTot'] = precipTotArray
    
    # add water year coordinate
    water_year = (tempArray.time.dt.month >= 10) + tempArray.time.dt.year
    stationXarray.coords['water_year'] = water_year

    if not for2526:
        # make sure you are only including potentially complete years
        if len(stationXarray.time)<366: # less than a complete year, catches stations that started in 2024 fall
            pass
        elif stationXarray.time[0] != datetime.datetime(water_year.min().values-1,10,1):
            stationXarray = stationXarray.sel(time=slice(f'{water_year.min().values}-10-01','2100'))
        elif stationXarray.time[0] == datetime.datetime(water_year.min().values-1,10,1):
            pass
        
        if stationXarray.time[-1] < datetime.datetime(water_year.max().values,7,15): # if the last WY has data through July 15, count it
            stationXarray = stationXarray.sel(time=slice(f'{stationXarray.time[0].dt.year.values}-10-01',f'{water_year.max().values-1}-09-30'))
  

    ## Add additional station attributes ##
    
    # get the station number from the dataPath
    statNum = float(dataPath[dataPath.index('_')+1:-4])
    stationMetaData = metaData.loc[metaData['stat_num'] == float(dataPath[dataPath.index('_')+1:-4])]
    
    stationLat = stationMetaData.lat.values[0]
    stationLon = stationMetaData.lon.values[0]
    stationElev_m = stationMetaData.elev.values[0]/3.281
    stationName = stationMetaData.site_name.values[0]
    
    stationXarray = stationXarray.assign_attrs(name=stationName,
                                               number=int(statNum),
                                               latitude=stationLat,
                                               longitude=stationLon,
                                               elevation_m=stationElev_m)
    
    ## Mark stations with bad data ##
        # data can be missing, largely incomplete, etc.
    if len(stationXarray.time)==0:
        stationXarray = stationXarray.assign_attrs(BAD_DATA = True)
    elif not (stationXarray.swe.max()>0):
        stationXarray = stationXarray.assign_attrs(BAD_DATA = True)
    else:
        stationXarray = stationXarray.assign_attrs(BAD_DATA = False)
    
    

    return stationXarray

    ### For adding dayofyear and year dimensions ###
    #stationXarray['dayofyear'] = tempArray.time.dt.dayofyear
    #stationXarray['year'] = tempArray.time.dt.year

    #stationXarray = stationXarray.set_index(time=['dayofyear','year']).unstack()

In [17]:
def loadStationMetaData():
    ''' This function loads station metadata into the environment '''
    
    stationMetaData = pd.read_excel('...../data/snotelStationNames.xlsx')
    
    return stationMetaData

In [18]:
def checkForEnoughData(waterYearArray,dataCat=None,fractionFull=.9,seasonalCutOff=[]):
    '''Check that at least fractionFull of the data in waterYearArray is not nan.
    
    Inputs:
        - waterYearArray: xarray that contains a water year's worth of data for a single variable (i.e. precip, swe, etc.)
        - dataCat: category of data in waterYearArray (options include "temp", "precip"). There may be additional checks for these data
        - fractionFull: required fraction of non-nan values
        - seasonalCutOff: indices if only looking a required fraction of non-nan values for a part of the year
    Returns:
        - Boolean indicating if threshold is met
    '''
    
    if len(seasonalCutOff)>0:
        numberOfNANs = waterYearArray[seasonalCutOff].isnull().sum().values
        bool1 = numberOfNANs/len(seasonalCutOff) < 1-fractionFull
    else:
        numberOfNANs = waterYearArray.isnull().sum().values
        bool1 = numberOfNANs/365 < 1-fractionFull
    
    bool2 = True
    if dataCat == 'temp': # additional temperature checks
        numberOfNANs = waterYearArray.isnull().sum().values
        if numberOfNANs == len(waterYearArray):
            bool2 = False
        else:
            maxT = np.nanmax(waterYearArray)
            if np.sum(waterYearArray == 0)>40: # too many 0s in the array
                bool2 = False
            elif maxT<0: # if daily mean temperature never gets above 0, then there is a problem...
                bool2 = False
            elif abs(maxT-12.722222222222221)<1e-5:
                if abs(np.nanmean(waterYearArray)- -7.228429546865301)<1e-3: 
                    # 2005 data in 'Trinchera (829) (C0)' is weird... this catches it
                    bool2 = False
    elif dataCat == 'swe':
        # check to make sure there is actually swe recorded
        indMax = np.argmax(waterYearArray.values)
        if waterYearArray.values[indMax]>0: # ensure that there is actually snow here
            pass
        else:
            bool2 = False
         
    boolReturn = bool1 and bool2
    return boolReturn

In [19]:
def loadTempBiasMarker():
    ''' Some SNOTEL stations have a temperature bias. This function returns a dictionary where the key is the
        station number and the values is a string indicating whether the temperature has been corrected.

        Returns:
            stationBiasDict
    '''
    
    stationBiasDict = {}
    
    stationBiasData = pd.read_csv('....../data/temperatureBiasData.txt', delimiter=',')
    stationTriplets = stationBiasData['stationTriplet'].values
        
    for stationTriplet in stationTriplets:
        stationRow = stationBiasData.loc[stationBiasData['stationTriplet'] == stationTriplet]
        
        stationNumber = stationTriplet[:stationTriplet.index(':')]
        
        stationBiasDict[int(stationNumber)] = stationRow['status'].values[0]
        
    return stationBiasDict

In [20]:
def fillNANTemp(tempTS):
    '''
    Use linear interpolation to fill in missing temperature values in observations that contain almost all of the days. In
    other words, the intent of this function is to fill in data when only a few observations are missing.
    
    Inputs:
        - tempTS: array of temperatures in a single water year
    Outputs:
        - tempTSInterp: array with missing temperatures interpolated linearly
    '''
    
    tempTSInterp = copy.deepcopy(tempTS)
    
    nans, x = nan_helper(tempTSInterp)
    tempTSInterp[nans]= np.interp(x(nans), x(~nans), tempTSInterp[~nans])
    return tempTSInterp

def nan_helper(y):
    """Helper to handle indices and logical indices of NaNs.

    Input:
        - y, 1d numpy array with possible NaNs
    Output:
        - nans, logical indices of NaNs
        - index, a function, with signature indices= index(logical_indices),
          to convert logical indices of NaNs to 'equivalent' indices
    Example:
        >>> # linear interpolation of NaNs
        >>> nans, x= nan_helper(y)
        >>> y[nans]= np.interp(x(nans), x(~nans), y[~nans])
    """

    return np.isnan(y), lambda z: z.nonzero()[0]

In [21]:
def fillNANPrcp(prcpTS):
    '''
    Replace missing precipitation values in observations that contain almost all of the days with 0. In
    other words, the intent of this function is to fill in data when only a few observations are missing.
    
    Inputs:
        - prcpTS: array of precipitation in a single water year
    Outputs:
        - prcpTSInterp: array with missing precip values replaced with 0
    '''
    prcpTSInterp = copy.deepcopy(prcpTS)
    
    nanBool = np.isnan(prcpTSInterp)
    
    prcpTSInterp[nanBool] = 0
    
    return prcpTSInterp

In [3]:
def filterStations(stateNames,allStateDict,winterInds=np.arange(259),minDataPercent=.95,numberYearsForCalc=10,stationBiasBoolDefault=True):
    '''
    Apply this function prior to performing any computations on the SWE data in order to ensure consistent stations
    across the different metrics.
    
    Inputs:
        - stateNames: list of state names for the calculation
        - allStateDict: snotel data for the stations in the states given in stateNames
        - winterInds: indices for narrowing the time window over which you determine if there is enough data for the calculation.
            The default is Oct 1-June 16 (or 15 for leap years). Set to np.arange(366) to consider the entire water year.
        - minDataPercent: minimum percentage of data required the winter average to be calculated. Default is 0.95.
        - numberYearsForCalc: int. Minimum number of years for the station slope to be calculated. Default is 10.
        - stationBiasBoolDefault: boolean. Decides whether stations with no indication of temperature bias correction should
            be included. Default is True.
               
    Outputs:
        - allStateDictFiltered: new dictionary with the same format as allStateDict, but includes only the stations that
            meet the criteria specified in this function.
            
    '''
    
    allStateDictFiltered = {}
    for iState,stateName in enumerate(stateNames):   
        currStateData = allStateDict[stateName]
        currStateDataNew = [] # list of filtered stations
        for iS,currStation in (enumerate(currStateData)):
            # some stations are processed before the next steps
            currStation = edgeCasesMod(currStation) 
            currStation = dropMin45C(currStation)
            goodWYList = []
            print(f'{stateNames[:iState+1]}: {iS}/{len(currStateData)}         ', end="\r")
            try:
                stationBiasBool = stationBiasDict[currStation.number]=='Complete'
            except KeyError:
                stationBiasBool = stationBiasBoolDefault
                #print('No bias data')
            if stationBiasBool: # only look at stations with corrected temp
                currStationWYs = np.unique(currStation.water_year.values)
                excludeMin70AndBelow = currStation.T_avg
                excludeMin70AndBelow[excludeMin70AndBelow<=-70] = np.nan # temperatures less than -70C are almost certainly an error
                currStation['T_avg'] = excludeMin70AndBelow
                for WY in currStationWYs:
                    currWY = currStation.sel(time=(currStation.water_year == WY))
                    
                    # mask out SWE changes greater than 200mm (per Evan 2019)
                    sweDiff = currWY.swe.diff(dim='time')
                    sweDiffBelow = abs(sweDiff)<200
                    currWY = xr.concat([currWY.isel(time=0),currWY.where(sweDiffBelow)],dim='time')

                    # only include years with sufficient data
                    check1 = checkForEnoughData(currWY.swe,seasonalCutOff=winterInds,fractionFull=minDataPercent,dataCat='swe')
                    check2 = checkForEnoughData(currWY.prcp,seasonalCutOff=winterInds,fractionFull=minDataPercent)
                    check3 = checkForEnoughData(currWY.T_avg,seasonalCutOff=winterInds,fractionFull=minDataPercent,dataCat='temp')
                    checkEdge = edgeCasesBool(currWY)
                    if check1 and check2 and check3 and checkEdge:
                        # IMPLEMENT PRECIP TOTAL CHECK HERE
                        currSWE = currWY.swe
                        currT = currWY.T_avg
                        currP = currWY.prcp
                        currTime = currWY.time
                        currTInterp = fillNANTemp(currT.values)
                        currPInterp = fillNANPrcp(currP.values)    
                        indMax = np.argmax(currSWE.values)
                        summedPrecip = np.sum(currPInterp[:indMax+1])
                        sweMax = np.max(currSWE.values)
                        if summedPrecip>=sweMax-25:
                            goodWYList = goodWYList + [WY]
                if len(goodWYList)>numberYearsForCalc:
                    currStateDataNew = currStateDataNew + [currStation.where(currStation.water_year.isin(goodWYList),drop=True)]                          
        allStateDictFiltered[stateName] = currStateDataNew
    return allStateDictFiltered

In [1]:
def edgeCasesBool(currWY):
    '''
    Is the current water year-station combination an edge case that needs to filtered out?
    
    If so, return False. Otherwise, return True.
    '''
    
    stationName = currWY.name
    
    if stationName == 'Silver Creek Divide (757)':
        if np.unique(currWY.water_year.values) in [2007, 2008, 2009, 2010, 2011]:
            boolToReturn = False
        else:
            boolToReturn = True
    elif stationName == 'Baldy (310)':
        if np.unique(currWY.water_year.values) in [1991,1992,1993,1994,1995,2006]:
            boolToReturn = False
        else:
            boolToReturn = True
    elif stationName == 'Railroad Overpass (710)': # station in Oregon where it barely snows
        boolToReturn = False
    elif stationName == 'Ute Creek (1005)':
        if np.unique(currWY.water_year.values) in [2001]:
            boolToReturn = False
        else:
            boolToReturn = True
    elif stationName == 'Tahoe City Cross (809)':
        if np.unique(currWY.water_year.values) in [1990,1991]:
            boolToReturn = False
        else:
            boolToReturn = True  
    elif stationName in ['Park Cone (680)']:
        if np.unique(currWY.water_year.values) in [1986,1987,1988,1989,1990,1991]:
            boolToReturn = False
        else:
            boolToReturn = True
    elif stationName == 'Casper Mtn. (389)':
        if np.unique(currWY.water_year.values) < 1988:
            boolToReturn = False
        else:
            boolToReturn = True
    elif stationName == 'Slumgullion (762)':
        if np.unique(currWY.water_year.values) in [1994,1995]:
            boolToReturn = False
        else:
            boolToReturn = True
    elif stationName == 'Rabbit Ears (709)':
        if np.unique(currWY.water_year.values) in [1993,1995]:
            boolToReturn = False
        else:
            boolToReturn = True
    elif stationName == 'Wild Basin (1042)':
        if np.unique(currWY.water_year.values) in [2009,2010,2012]:
            boolToReturn = False
        else:
            boolToReturn = True
    elif stationName == 'Hourglass Lake (1122)':
        if np.unique(currWY.water_year.values) in [2010]:
            boolToReturn = False
        else:
            boolToReturn = True
    elif stationName == 'Moraine (1035)':
        if np.unique(currWY.water_year.values) in [2005,2020]: # hard to tell how bad 2020 is, may actually just be colder
            boolToReturn = False
        else:
            boolToReturn = True
    elif stationName == 'Heber (519)':
        if np.unique(currWY.water_year.values) in [2003]:
            boolToReturn = False
        else:
            boolToReturn = True
    elif stationName == 'Hannagan Meadows (511)':
        if np.unique(currWY.water_year.values) in [1989]:
            boolToReturn = False
        else:
            boolToReturn = True
    elif stationName == 'Mt. Ryan (948)':
        if np.unique(currWY.water_year.values) in [1989,1999,2000]:
            boolToReturn = False
        else:
            boolToReturn = True
    elif stationName in ['Hemlock Butte (520)','South Mtn. (774)']:
        if np.unique(currWY.water_year.values) in [1985]:
            boolToReturn = False
        else:
            boolToReturn = True
    elif stationName in ['Clear Lake (401)','Columbine Pass (409)']:
        if np.unique(currWY.water_year.values) in [1991]:
            boolToReturn = False
        else:
            boolToReturn = True
    elif stationName == 'Saint Elmo (1100)':
        if np.unique(currWY.water_year.values) in [2012,2013]:
            boolToReturn = False
        else:
            boolToReturn = True
    elif stationName == 'Stump Lakes (797)':
        if np.unique(currWY.water_year.values) in [1993,2002,2003,2004,2005]:
            boolToReturn = False
        else:
            boolToReturn = True
    elif stationName == 'Happy Jack (969)':
        if np.unique(currWY.water_year.values) in [2001,2007,2008]:
            boolToReturn = False
        else:
            boolToReturn = True
    elif stationName == 'Brooklyn Lake (367)':
        if np.unique(currWY.water_year.values) in [1988,1989,1990,1991,1992]:
            boolToReturn = False
        else:
            boolToReturn = True
    elif stationName == 'Divide Peak (449)':
        if np.unique(currWY.water_year.values) in [1998,1997]:
            boolToReturn = False
        else:
            boolToReturn = True
    elif stationName == 'Tolby (934)':
        if np.unique(currWY.water_year.values) in [2009,2019]:
            boolToReturn = False
        else:
            boolToReturn = True
    elif stationName in ['Promontory (705)','Fry (488)']:
        if np.unique(currWY.water_year.values) < 1994:
            boolToReturn = False
        else:
            boolToReturn = True
    elif stationName == 'Lone Pine (591)':
        if np.unique(currWY.water_year.values) in [1988]:
            boolToReturn = False
        else:
            boolToReturn = True
    elif stationName == 'Mormon Mountain (640)':
        if np.unique(currWY.water_year.values) in [2002,2003]:
            boolToReturn = False
        else:
            boolToReturn = True
    elif stationName == 'Blue Lakes (356)':
        if np.unique(currWY.water_year.values) in [1984]:
            boolToReturn = False
        else:
            boolToReturn = True
    elif stationName == 'Holland Meadows (529)':
        if np.unique(currWY.water_year.values) in [1985]:
            boolToReturn = False
        else:
            boolToReturn = True
    elif stationName == 'Togwotee Pass (822)':
        if np.unique(currWY.water_year.values) in [1983]:
            boolToReturn = False
        else:
            boolToReturn = True
    else:
        boolToReturn = True
        
    return boolToReturn

In [1]:
def edgeCasesMod(allYears):
    '''
    Modify the certain station data prior to checking if there is sufficient data.
    '''
    
    stationName = allYears.name
    
    if stationName == 'Moraine (1035)':
        currT = allYears.T_avg
        currT[currT<-500] = np.nan
        allYears['T_avg'] = currT
        return allYears
    elif stationName == 'Park Reservoir (682)':
        currT = allYears.T_avg
        currT[currT<-40] = np.nan
        allYears['T_avg'] = currT
        return allYears
    elif stationName == 'Berthoud Summit (335)':
        currT = allYears.T_avg
        currT[currT<-40] = np.nan
        allYears['T_avg'] = currT
        return allYears
    elif stationName in ['Baker Butte (308)','Heber (519)','Hemlock Butte (520)']:
        currT = allYears.T_avg
        currT[currT<-25] = np.nan
        allYears['T_avg'] = currT
        return allYears
    elif stationName in ['Brooklyn Lake (367)','Casper Mtn. (389)']:
        currT = allYears.T_avg
        currT[currT<-29] = np.nan
        allYears['T_avg'] = currT
        return allYears
    elif stationName in ['Lizard Head Pass (586)']:
        currT = allYears.T_avg
        currT[currT<-23] = np.nan
        allYears['T_avg'] = currT
        return allYears
    elif stationName in ['Lone Pine (591)']:
        currT = allYears.T_avg
        currT[currT<-15] = np.nan
        allYears['T_avg'] = currT
        return allYears
    else:
        return allYears

In [25]:
def dropMin45C(allYears):
    '''
    Many stations have an error where the temperature is at or below -45C. Set these values to NAN.
    '''
    stationName = allYears.name
    
    stationsWith50C = ['Rabbit Ears (709)','White Horse Lake (861)','North French Creek (668)',
                       'Slumgullion (762)','Berthoud Summit (335)','Elk River (467)','Promontory (705)',
                       'Fry (488)','Hannagan Meadows (511)','Independence Camp (539)','Mt Rose Ski Area (652)'
                       'Salt Creek Falls (729)','Brooklyn Lake (367)','South Brush Creek (772)','Summit Ranch (802)',
                      'Columbine Pass (409)']
                       
                       
    if True:#stationName in stationsWith50C: # has many erroneous temperature measurements (recorded at <=-45C)
        currT = allYears.T_avg
        currT[currT<=-45] = np.nan
        allYears['T_avg'] = currT
        return allYears
    else:
        return allYears

In [1]:
def stationLookupByNumber(statNum,allStateDict):
    '''
    Return the station xarray with number given in statNum
    '''
    for iState,stateName in enumerate(stateNames):   
        currStateData = allStateDict[stateName]
        for iS,currStation in (enumerate(currStateData)):
            if currStation.number == statNum:
                return currStation
    print('Number not found!')
    return False

# Calculations on data
Note: all of these functions should be run after "filterStations" is called on the SWE dictionary

## Metrics of integrated prcp vs SWE

### frozenDayPrcpCalc

In [26]:
def frozenDayPrcpCalc(stateNames, allStateDict, snowTempThresh=0):
    '''
    Calculate the precipitation on days with daily mean temperature less than snowTempThresh and 
    peak SWE for the states and stations given in stateNames and allStateDict.
    
    Inputs:
        stateNames: list of state names for the calculation
        allStateDict: snotel data for the stations in the states given in stateNames
        snowTempThresh: temperature threshold (in Celcius) below which the precipitation is integrated. Default value is 0.
    
    Outputs:
        allStateDictNew: new dictionary with the same format as allStateDict, but with modified attributes
        to include frozenDayPrcp and peakSWE
    '''
    
    allStateDictNew = {}
    for iState,stateName in enumerate(stateNames):   
        currStateData = allStateDict[stateName]
        currStateDataNew = [] # list of modified stations
        for iS,currStation in (enumerate(currStateData)):
            print(f'{stateNames[:iState+1]}: {iS}/{len(currStateData)}         ', end="\r")
            currStationWYs = np.unique(currStation.water_year.values)
            peakSWE = []
            intPrcp = []
            for WY in currStationWYs:
                currWY = currStation.sel(time=(currStation.water_year == WY))
                currSWE = currWY.swe
                currT = currWY.T_avg
                currP = currWY.prcp          
                currTInterp = fillNANTemp(currT.values)
                currPInterp = fillNANPrcp(currP.values)                 
                peakSWE = peakSWE + [currSWE.max().values]
                intPrcp = intPrcp + [currPInterp[currTInterp<snowTempThresh].sum()]
            currStation = currStation.assign_attrs(frozenDayPrcp = intPrcp)
            currStation = currStation.assign_attrs(peakSWE = peakSWE)
            currStateDataNew = currStateDataNew + [currStation]                          
        allStateDictNew[stateName] = currStateDataNew
    return allStateDictNew

### intPrcpSWESlopeCalc

In [27]:
def intPrcpSWESlopeCalc(stateNames, allStateDict, snowTempThresh=0):
    '''
    Calculate the slope between precipitation on days with daily mean temperature less than snowTempThresh and 
    peak SWE for the states and stations given in stateNames and allStateDict. Slopes less than 1 
    indicate that in total more precipitation fell on days below snowTempThresh when compared to the maximum SWE observed in that year,
    or that there was sufficient melting so that sum(P)_{T<0} > max(SWE).
    
    Inputs:
        stateNames: list of state names for the calculation
        allStateDict: snotel data for the stations in the states given in stateNames
        snowTempThresh: temperature threshold (in Celcius) below which the precipitation is integrated. Default value is 0.
    
    Outputs:
        allStateDictNew: new dictionary with the same format as allStateDict, but with modified attributes
        to include intPrcpSWESlope
    '''
    
    allStateDictNew = {}
    for iState,stateName in enumerate(stateNames):   
        currStateData = allStateDict[stateName]
        currStateDataNew = [] # list of modified stations
        for iS,currStation in (enumerate(currStateData)):
            print(f'{stateNames[:iState+1]}: {iS}/{len(currStateData)}         ', end="\r")
            currStationWYs = np.unique(currStation.water_year.values)
            peakSWE = []
            intPrcp = []
            for WY in currStationWYs:
                currWY = currStation.sel(time=(currStation.water_year == WY))
                currSWE = currWY.swe
                currT = currWY.T_avg
                currP = currWY.prcp          
                currTInterp = fillNANTemp(currT.values)
                currPInterp = fillNANPrcp(currP.values)                 
                peakSWE = peakSWE + [currSWE.max().values]
                intPrcp = intPrcp + [currPInterp[currTInterp<snowTempThresh].sum()]
            slope, intercept, r_value, p_value, std_err = scipy.stats.linregress(np.array(intPrcp), np.array(peakSWE))
            currStation = currStation.assign_attrs(intPrcpSWESlope = slope)
            currStation = currStation.assign_attrs(intPrcpSWER2Max = r_value**2)
            currStateDataNew = currStateDataNew + [currStation]             
            # diagnostics
            if slope<.5:
                print(f'\n{stateName}: {currStation.name}, slope = {round(slope,3)}'+'\n')             
        allStateDictNew[stateName] = currStateDataNew
    return allStateDictNew

### intPrcpSWER2Calc

In [28]:
def intPrcpSWER2Calc(stateNames, allStateDict, snowTempThresh=0):
    '''
    Calculate the R2 from a 1-1 line between precipitation on days with daily mean temperature less than snowTempThresh and 
    peak SWE for the states and stations given in stateNames and allStateDict.
    
    Inputs:
        stateNames: list of state names for the calculation
        allStateDict: snotel data for the stations in the states given in stateNames
        snowTempThresh: temperature threshold (in Celcius) below which the precipitation is integrated. Default value is 0.

    Outputs:
        allStateDictNew: new dictionary with the same format as allStateDict, but with modified attributes
        to include intPrcpSWER2
    '''
    
    allStateDictNew = {}
    for iState,stateName in enumerate(stateNames):   
        currStateData = allStateDict[stateName]
        currStateDataNew = [] # list of modified stations
        for iS,currStation in (enumerate(currStateData)):
            print(f'{stateNames[:iState+1]}: {iS}/{len(currStateData)}         ', end="\r")
            currStationWYs = np.unique(currStation.water_year.values)
            peakSWE = []
            intPrcp = []
            for WY in currStationWYs:
                currWY = currStation.sel(time=(currStation.water_year == WY))
                currSWE = currWY.swe
                currT = currWY.T_avg
                currP = currWY.prcp          
                currTInterp = fillNANTemp(currT.values)
                currPInterp = fillNANPrcp(currP.values)                 
                peakSWE = peakSWE + [currSWE.max().values]
                intPrcp = intPrcp + [currPInterp[currTInterp<snowTempThresh].sum()]
            R2_1 = R2_from1to1(np.array(peakSWE),np.array(intPrcp))
            R2_2 = R2_from1to1(np.array(intPrcp),np.array(peakSWE))
            R2 = max(R2_1,R2_2)
            currStation = currStation.assign_attrs(intPrcpSWER2 = R2)
            currStateDataNew = currStateDataNew + [currStation]             
            # diagnostics
            # if R2<.05:
                #print(f'\n{stateName}: {currStation.name}, R^2 = {round(R2,3)}'+'\n')             
        allStateDictNew[stateName] = currStateDataNew
    return allStateDictNew

### winterPrcpSWESlopeCalc

In [29]:
def winterPrcpSWESlopeCalc(stateNames, allStateDict):
    '''
    Calculate the slope between integrated precipitation between Nov and Mar and
    peak SWE for the states and stations given in stateNames and allStateDict. Slopes less than 1 
    indicate that in total more precipitation fell between Nov and Mar when compared to the maximum SWE observed in that year.
    
    Inputs:
        stateNames: list of state names for the calculation
        allStateDict: snotel data for the stations in the states given in stateNames
        snowTempThresh: temperature threshold (in Celcius) below which the precipitation is integrated. Default value is 0.
    Outputs:
        allStateDictNew: new dictionary with the same format as allStateDict, but with modified attributes
        to include winterPrcpSWESlope
    '''
    
    allStateDictNew = {}
    for iState,stateName in enumerate(stateNames):   
        currStateData = allStateDict[stateName]
        currStateDataNew = [] # list of modified stations
        for iS,currStation in (enumerate(currStateData)):
            print(f'{stateNames[:iState+1]}: {iS}/{len(currStateData)}         ', end="\r")
            currStationWYs = np.unique(currStation.water_year.values)
            peakSWE = []
            intPrcp = []
            for WY in currStationWYs:
                currWY = currStation.sel(time=(currStation.water_year == WY))
                currSWE = currWY.swe
                currT = currWY.T_avg
                currP = currWY.prcp
                currTime = currWY.time               
                novThruMarInds = np.logical_or(currTime.dt.month >= 11,currTime.dt.month <= 3)              
                currTInterp = fillNANTemp(currT.values)
                currPInterp = fillNANPrcp(currP.values)             
                peakSWE = peakSWE + [currSWE.max().values]
                intPrcp = intPrcp + [currPInterp[novThruMarInds].sum()]
            slope, intercept, r_value, p_value, std_err = scipy.stats.linregress(np.array(intPrcp), np.array(peakSWE))
            currStation = currStation.assign_attrs(winterPrcpSWESlope = slope)
            currStation = currStation.assign_attrs(winterPrcpSWER2Max = r_value**2)
            currStateDataNew = currStateDataNew + [currStation]        
            # diagnostics
            if slope<.5:
                print(f'\n{stateName}: {currStation.name}, slope = {round(slope,3)}\n')
        allStateDictNew[stateName] = currStateDataNew
    return allStateDictNew

### winterPrcpSWER2Calc

In [ ]:
def winterPrcpSWER2Calc(stateNames, allStateDict, winterInds=np.arange(259), numberYearsForCalc=7, stationBiasBoolDefault=True):
    '''
    Calculate the R2 for a 1-1 line using integrated precipitation between Nov and Mar and
    peak SWE for the states and stations given in stateNames and allStateDict.
    
    Inputs:
        stateNames: list of state names for the calculation
        allStateDict: snotel data for the stations in the states given in stateNames
        snowTempThresh: temperature threshold (in Celcius) below which the precipitation is integrated. Default value is 0.
        winterInds: indices for narrowing the time window over which you determine if there is enough data for the calculation.
            The default is Oct 1-June 16 (or 15 for leap years). Set to np.arange(366) to consider the entire water year.
        numberYearsForCalc: int. Minimum number of years for the station slope to be calculated. Default is 7.
        stationBiasBoolDefault: boolean. Decides whether stations with no indication of temperature bias correction should
            be included. Default is True.
    
    Outputs:
        allStateDictNew: new dictionary with the same format as allStateDict, but with modified attributes
        to include winterPrcpSWER2
    '''
    
    allStateDictNew = {}
    for iState,stateName in enumerate(stateNames):   
        currStateData = allStateDict[stateName]
        currStateDataNew = [] # list of modified stations
        for iS,currStation in (enumerate(currStateData)):
            print(f'{stateNames[:iState+1]}: {iS}/{len(currStateData)}         ', end="\r")
            currStationWYs = np.unique(currStation.water_year.values)
            peakSWE = []
            intPrcp = []
            for WY in currStationWYs:
                currWY = currStation.sel(time=(currStation.water_year == WY))    
                currSWE = currWY.swe
                currT = currWY.T_avg
                currP = currWY.prcp
                currTime = currWY.time               
                novThruMarInds = np.logical_or(currTime.dt.month >= 11,currTime.dt.month <= 3)              
                currTInterp = fillNANTemp(currT.values)
                currPInterp = fillNANPrcp(currP.values)             
                peakSWE = peakSWE + [currSWE.max().values]
                intPrcp = intPrcp + [currPInterp[novThruMarInds].sum()]
            R2_1 = R2_from1to1(np.array(peakSWE),np.array(intPrcp))
            R2_2 = R2_from1to1(np.array(intPrcp),np.array(peakSWE))
            R2 = max(R2_1,R2_2)
            currStation = currStation.assign_attrs(winterPrcpSWER2 = R2)
            currStateDataNew = currStateDataNew + [currStation]     
            # diagnostics
            # if R2<.05:
                #print(f'\n{stateName}: {currStation.name}, R^2 = {round(R2,3)}'+'\n')        
        allStateDictNew[stateName] = currStateDataNew
    return allStateDictNew

## Using the simple SWE model

### modelObsR2Calc

In [5]:
def modelObsR2Calc(stateNames,allStateDict,k,accumThresh=0):
    '''
    Calculate the R^2 value (for a 1-1 fit) using the annual maximum SWE from the simple SWE model and the observations. For each
    water year in each station in each state, determine the peak SWE from the observations. Compare this to the peak swe predicted
    by the model when using the observed meteorology as inputs.
    
    Inputs:
        stateNames: list of state names for the calculation
        allStateDict: snotel data for the stations in the states given in stateNames
        k: melt rate coefficient in the model. This is the same for all stations in this function. (units of mm/(degree day))
        accumThresh: temperature threshold (in degree C) below which SWE accumulates (default is 0C). This is the same for all 
            stations in this function.
      Outputs:
        allStateDictNew: new dictionary with the same format as allStateDict, but with modified attributes
        to include modelObsR2
    '''
    
    allStateDictNew = {}
    for iState,stateName in enumerate(stateNames):   
        currStateData = allStateDict[stateName]
        currStateDataNew = [] # list of modified stations
        for iS,currStation in (enumerate(currStateData)):
            print(f'{stateNames[:iState+1]}: {iS}/{len(currStateData)}         ', end="\r")
            currStationWYs = np.unique(currStation.water_year.values)
            peakSWEObs = np.array([])
            peakSWEModel = np.array([])
            for WY in currStationWYs:
                currWY = currStation.sel(time=(currStation.water_year == WY))
                currT = currWY.T_avg
                currP = currWY.prcp         
                currTInterp = fillNANTemp(currT.values)
                currPInterp = fillNANPrcp(currP.values)            
                currSWE_mod = sweModel(currTInterp,currPInterp,k,accumThresh)                
                # check that there is enough precip to reasonably account for the SWE in the obs
                indMaxSWEObs = currWY.swe.argmax().values
                if currWY.prcp[:indMaxSWEObs].sum() > currWY.swe.max().values - 100: # within 100 mm             
                    peakSWEObs = np.append(peakSWEObs,currWY.swe.max().values)
                    peakSWEModel = np.append(peakSWEModel,np.nanmax(currSWE_mod))
            R2_1 = R2_from1to1(peakSWEObs,peakSWEModel)
            R2_2 = R2_from1to1(peakSWEModel,peakSWEObs)
            R2 = max(R2_1,R2_2)           
            #if R2<0:
            #    R2 = 0
            if R2<.05:
                print(f'\n{stateName}: {currStation.name}'+'\n')
            currStation = currStation.assign_attrs(modelObsR2 = R2)
            currStateDataNew = currStateDataNew + [currStation]
        allStateDictNew[stateName] = currStateDataNew
    return allStateDictNew

### modelObsSlopeCalc

In [6]:
def modelObsSlopeCalc(stateNames,allStateDict,k,accumThresh=0):
    '''
    Calculate the slope the maximizes the R2 value using the annual maximum SWE from the simple SWE model and the observations. For each
    water year in each station in each state, determine the peak SWE from the observations. Compare this to the peak swe predicted
    by the model when using the observed meteorology as inputs.
    
    Inputs:
        stateNames: list of state names for the calculation
        allStateDict: snotel data for the stations in the states given in stateNames
        k: melt rate coefficient in the model. This is the same for all stations in this function. (units of mm/(degree day))
        accumThresh: temperature threshold (in degree C) below which SWE accumulates (default is 0C). This is the same for all 
            stations in this function.
    Outputs:
        allStateDictNew: new dictionary with the same format as allStateDict, but with modified attributes
        to include modelObsR2
    '''
    
    allStateDictNew = {}
    for iState,stateName in enumerate(stateNames):   
        currStateData = allStateDict[stateName]
        currStateDataNew = [] # list of modified stations
        for iS,currStation in (enumerate(currStateData)):
            print(f'{stateNames[:iState+1]}: {iS}/{len(currStateData)}         ', end="\r")
            currStationWYs = np.unique(currStation.water_year.values)
            peakSWEObs = np.array([])
            peakSWEModel = np.array([])
            for WY in currStationWYs:
                currWY = currStation.sel(time=(currStation.water_year == WY))
                currT = currWY.T_avg
                currP = currWY.prcp         
                currTInterp = fillNANTemp(currT.values)
                currPInterp = fillNANPrcp(currP.values)            
                currSWE_mod = sweModel(currTInterp,currPInterp,k,accumThresh)                
                # check that there is enough precip to reasonably account for the SWE in the obs
                indMaxSWEObs = currWY.swe.argmax().values
                if currWY.prcp[:indMaxSWEObs].sum() > currWY.swe.max().values - 100: # within 100 mm             
                    peakSWEObs = np.append(peakSWEObs,currWY.swe.max().values)
                    peakSWEModel = np.append(peakSWEModel,np.nanmax(currSWE_mod))
            slope, intercept, r_value, p_value, std_err = scipy.stats.linregress(peakSWEObs, peakSWEModel)
            currStation = currStation.assign_attrs(modelObsSlope = slope)
            currStation = currStation.assign_attrs(modelObsR2Max = r_value**2)
            currStateDataNew = currStateDataNew + [currStation]
        allStateDictNew[stateName] = currStateDataNew
    return allStateDictNew

### dateOfSWEMetricModel

In [7]:
def dateOfSWEMetricModel(stateNames,allStateDict,metricName,k,accumThresh=0):
    '''
    Calculate date of a specified metric of SWE (meltOut, peakSWE, etc.) in the simple model.
    
    Inputs:
        - stateNames: list of state names for the calculation
        - allStateDict: snotel data for the stations in the states given in stateNames
        - metricName: string describing the metric name
        - k: melt rate coefficient in the model. This is the same for all stations in this function. (units of mm/(degree day))
        - accumThresh: temperature threshold (in degree C) below which SWE accumulates (default is 0C). This is the same for all 
            stations in this function.
    
    Outputs:
        - allStateDictNew: new dictionary with the same format as allStateDict, but with modified attributes
        to include metricName
    '''
    
    allStateDictNew = {}  
    for iState,stateName in enumerate(stateNames):        
        currStateData = allStateDict[stateName]
        currStateDataNew = [] # list of modified stations
        for iS,currStation in (enumerate(currStateData)):
            print(f'{stateNames[:iState+1]}: {iS}/{len(currStateData)}         ', end="\r")
            currStationWYs = np.unique(currStation.water_year.values)
            sweMetricAllYears = np.array([],dtype='datetime64')
            sweMetricYears = np.array([])
            for WY in currStationWYs:
                currWY = currStation.sel(time=(currStation.water_year == WY))
                currT = currWY.T_avg
                currP = currWY.prcp         
                currTInterp = fillNANTemp(currT.values)
                currPInterp = fillNANPrcp(currP.values)            
                currSWE_mod = sweModel(currTInterp,currPInterp,k,accumThresh)  
                currTime = currWY.time
                if metricName=='peakSWE':
                    indMax = np.argmax(currSWE_mod)
                    dayOfMax = currTime[indMax]
                    sweMetricAllYears = np.append(sweMetricAllYears,dayOfMax)
                sweMetricYears = np.append(sweMetricYears,WY)
            currStation = currStation.assign_attrs({metricName+'Mod':sweMetricAllYears,metricName+'Years':sweMetricYears})
            currStateDataNew = currStateDataNew + [currStation]
        allStateDictNew[stateName] = currStateDataNew
    return allStateDictNew
    

### modelObsPeakSWE

In [8]:
def modelObsPeakSWE(stateNames,allStateDict,k,accumThresh=0):
    '''
    Calculate the peak SWE in the simple model and compare with observations.
    
    Inputs:
        - stateNames: list of state names for the calculation
        - allStateDict: snotel data for the stations in the states given in stateNames
        - k: melt rate coefficient in the model. This is the same for all stations in this function. (units of mm/(degree day))
        - accumThresh: temperature threshold (in degree C) below which SWE accumulates (default is 0C). This is the same for all 
            stations in this function.
    
    Outputs:
        - allStateDictNew: new dictionary with the same format as allStateDict, but with modified attributes
        to include peakSWEsObs and peakSWEsModel
    '''
    
    allStateDictNew = {}  
    for iState,stateName in enumerate(stateNames):        
        currStateData = allStateDict[stateName]
        currStateDataNew = [] # list of modified stations
        for iS,currStation in (enumerate(currStateData)):
            print(f'{stateNames[:iState+1]}: {iS}/{len(currStateData)}         ', end="\r")
            # only look at stations with corrected temp
            currStationWYs = np.unique(currStation.water_year.values)
            peakSWEsModel = np.array([])
            peakSWEsObs = np.array([])
            for WY in currStationWYs:
                currWY = currStation.sel(time=(currStation.water_year == WY))
                currT = currWY.T_avg
                currP = currWY.prcp
                currSWE = currWY.swe
                currTInterp = fillNANTemp(currT.values)
                currPInterp = fillNANPrcp(currP.values)            
                currSWE_mod = sweModelVec(currTInterp,currPInterp,k,accumThresh)
                peakSWEsModel = np.append(peakSWEsModel,np.nanmax(currSWE_mod))
                peakSWEsObs = np.append(peakSWEsObs,currWY.swe.max().values)
            currStation = currStation.assign_attrs({'peakSWEsObs':peakSWEsObs,'peakSWEsModel':peakSWEsModel})
            currStateDataNew = currStateDataNew + [currStation]
        allStateDictNew[stateName] = currStateDataNew
    return allStateDictNew

## General SWE climatology

### obsMeanWinterTemp

In [9]:
def obsMeanWinterTemp(stateNames,allStateDict):
    '''
    Calculate the mean winter (Nov-March) temperature for each station with valid and sufficient temperature data.
    
    Inputs:
        - stateNames: list of state names for the calculation
        - allStateDict: snotel data for the stations in the states given in stateNames
            
    Outputs:
        - allStateDictNew: new dictionary with the same format as allStateDict, but with modified attributes
        to include meanWinterTemp
    '''
    
    allStateDictNew = {}
    for iState,stateName in enumerate(stateNames):
        currStateData = allStateDict[stateName]
        currStateDataNew = [] # list of modified stations
        for iS,currStation in (enumerate(currStateData)):
            print(f'{stateNames[:iState+1]}: {iS}/{len(currStateData)}         ', end="\r")
            currStationWYs = np.unique(currStation.water_year.values)
            meanWinterTempAllYears = np.array([],)
            for WY in currStationWYs:
                currWY = currStation.sel(time=(currStation.water_year == WY))
                currT = currWY.T_avg
                currTime = currWY.time
                # remove any obviously erroneous temperature values (i.e. way to warm)
                currT[currT>40] = np.nan
                currTInterp = fillNANTemp(currT.values)
                winterInds = np.logical_or(currTime.dt.month >= 11,currTime.dt.month <= 3)
                meanWinterTemp = np.nanmean(currTInterp[winterInds])
                meanWinterTempAllYears = np.append(meanWinterTempAllYears,meanWinterTemp)
            currStation = currStation.assign_attrs(meanWinterTemp = np.mean(meanWinterTempAllYears))
            currStateDataNew = currStateDataNew + [currStation]
        allStateDictNew[stateName] = currStateDataNew
    return allStateDictNew

### dateOfSWEMetric

In [10]:
def dateOfSWEMetricObs(stateNames,allStateDict,metricName):
    '''
    Calculate date of a specified metric of SWE in the observations(meltOut, peakSWETiming, etc.).
    
    Inputs:
        - stateNames: list of state names for the calculation
        - allStateDict: snotel data for the stations in the states given in stateNames
        - metricName: string describing the metric name
            
    Outputs:
        - allStateDictNew: new dictionary with the same format as allStateDict, but with modified attributes
        to include metricName
    '''
    
    allStateDictNew = {}  
    for iState,stateName in enumerate(stateNames):        
        currStateData = allStateDict[stateName]
        currStateDataNew = [] # list of modified stations
        for iS,currStation in (enumerate(currStateData)):
            print(f'{stateNames[:iState+1]}: {iS}/{len(currStateData)}         ', end="\r")
            currStationWYs = np.unique(currStation.water_year.values)
            sweMetricAllYears = np.array([],dtype='datetime64')
            sweMetricYears = np.array([])
            for WY in currStationWYs:
                currWY = currStation.sel(time=(currStation.water_year == WY))
                currSWE = currWY.swe.values
                currTime = currWY.time
                if metricName=='peakSWETiming':
                    indMax = np.argmax(currSWE)
                    dayOfMax = currTime[indMax]
                    if currSWE[indMax]>0: # ensure that there is actually snow here
                        sweMetricAllYears = np.append(sweMetricAllYears,dayOfMax)
                elif metricName=='snowStart': # time of first snowfall
                    for i in range(len(currSWE)-10):
                        if np.sum(currSWE[i:i+10]!=0) == 10:
                            firstSnowInd = i
                            break
                    dayOfFirstSnow = currTime[firstSnowInd]
                    indMax = np.argmax(currSWE)
                    dayOfMax = currTime[indMax]
                    if currSWE[indMax]>0: # ensure that there is actually snow here
                        sweMetricAllYears = np.append(sweMetricAllYears,dayOfFirstSnow)
                else:
                    print('Oops, invalid metric!')
                    return
                sweMetricYears = np.append(sweMetricYears,WY)
            currStation = currStation.assign_attrs({metricName:sweMetricAllYears,metricName+'Years':sweMetricYears})
            currStateDataNew = currStateDataNew + [currStation]
        allStateDictNew[stateName] = currStateDataNew
    return allStateDictNew

### obsMeanWinterTotPrcp

In [11]:
def obsMeanWinterTotPrcp(stateNames,allStateDict):
    '''
    Calculate the mean winter (Nov-March) cumulative precipitation for each station with valid and sufficient precip data.
    
    Inputs:
        - stateNames: list of state names for the calculation
        - allStateDict: snotel data for the stations in the states given in stateNames
    Outputs:
        - allStateDictNew: new dictionary with the same format as allStateDict, but with modified attributes
        to include meanWinterPrcpTot
    '''
    
    allStateDictNew = {}  
    for iState,stateName in enumerate(stateNames):        
        currStateData = allStateDict[stateName]
        currStateDataNew = [] # list of modified stations
        for iS,currStation in (enumerate(currStateData)):
            print(f'{stateNames[:iState+1]}: {iS}/{len(currStateData)}         ', end="\r")
            currStationWYs = np.unique(currStation.water_year.values)
            winterTotalPrcpAllYears = np.array([])
            for WY in currStationWYs:
                currWY = currStation.sel(time=(currStation.water_year == WY))
                currP = currWY.prcp
                currTime = currWY.time
                currPInterp = fillNANPrcp(currP.values)
                winterInds = np.logical_or(currTime.dt.month >= 11,currTime.dt.month <= 3)
                currTotalPrcp = np.nansum(currPInterp[winterInds])
                winterTotalPrcpAllYears = np.append(winterTotalPrcpAllYears,currTotalPrcp)
            currStation = currStation.assign_attrs(meanWinterPrcpTot = np.nanmean(winterTotalPrcpAllYears))
            currStateDataNew = currStateDataNew + [currStation]
        allStateDictNew[stateName] = currStateDataNew
    return allStateDictNew

## Prcp rate and freezing season length

### prcpRateTimesFreezingTime

In [12]:
def prcpRateTimesFreezingTime(stateNames,allStateDict,accumThresh=0):
    '''
    Calculate mean winter precipitation rate times the number of days below freezing
    (or another specified threshold accumThresh) and compare to max SWE.
    
    Inputs:
        - stateNames: list of state names for the calculation
        - allStateDict: snotel data for the stations in the states given in stateNames
        - accumThresh: temperature threshold (in degree C) below which SWE accumulates (default is 0C). This is the same for all 
            stations in this function.            
    Outputs:
        - allStateDictNew: new dictionary with the same format as allStateDict, but with modified attributes
        to include precipTimesFreezeTime
    '''
    
    allStateDictNew = {}  
    for iState,stateName in enumerate(stateNames):        
        currStateData = allStateDict[stateName]
        currStateDataNew = [] # list of modified stations
        for iS,currStation in (enumerate(currStateData)):
            print(f'{stateNames[:iState+1]}: {iS}/{len(currStateData)}         ', end="\r")
            currStationWYs = np.unique(currStation.water_year.values)
            obsPeakSWEs = np.array([])
            precipTimesFreezeTimes = np.array([])
            for WY in currStationWYs:
                currWY = currStation.sel(time=(currStation.water_year == WY))
                currT = currWY.T_avg
                currP = currWY.prcp         
                currTInterp = fillNANTemp(currT.values)
                currPInterp = fillNANPrcp(currP.values)            
                currTime = currWY.time
                currSWE = currWY.swe
                winterInds = np.logical_or(currTime.dt.month >= 11,currTime.dt.month <= 3)
                avgPrcpRate = np.nanmean(currPInterp[winterInds])
                numDaysBelowTThresh = np.sum(currTInterp<accumThresh)
                precipTimesFreezeTimes = np.append(precipTimesFreezeTimes,avgPrcpRate*numDaysBelowTThresh)
                obsPeakSWEs = np.append(obsPeakSWEs,currSWE.max().values)
            currStation = currStation.assign_attrs({'precipTimesFreezeTime':precipTimesFreezeTimes,'peakSWE':obsPeakSWEs})
            currStateDataNew = currStateDataNew + [currStation]
        allStateDictNew[stateName] = currStateDataNew
    return allStateDictNew  
                        

### winterPrcpRateByYear

In [2]:
def winterPrcpRateByYear(stateNames,allStateDict,accumThresh=0):
    '''
    Calculate mean winter precipitation rate in each year and number of days below accumThresh for every station.
    
    Inputs:
        - stateNames: list of state names for the calculation
        - allStateDict: snotel data for the stations in the states given in stateNames
        - accumThresh: temperature threshold (in degree C) below which SWE accumulates (default is 0C). This is the same for all 
            stations in this function.
                
    Outputs:
        - allStateDictNew: new dictionary with the same format as allStateDict, but with modified attributes
        to include precipRates (winter avg prcp rate), numDaysBelowTThresh_s, peakSWE,
        precipRates_duringSnow (mean precip rate during snow covered period), prcpDuringTauF 
        (mean precip on days below 0C), longestFreezeStreak, and Tw.
    '''
    
    allStateDictNew = {}  
    for iState,stateName in enumerate(stateNames):        
        currStateData = allStateDict[stateName]
        currStateDataNew = [] # list of modified stations
        for iS,currStation in (enumerate(currStateData)):
            print(f'{stateNames[:iState+1]}: {iS}/{len(currStateData)}         ', end="\r")
            currStationWYs = np.unique(currStation.water_year.values)
            precipRates = np.array([])
            numDaysBelowTThresh_s = np.array([])
            obsPeakSWEs = np.array([])
            precipRates_duringSnow = np.array([])
            prcpDuringTauF = np.array([])
            longestFreezeStreak = np.array([])
            Tw = np.array([])
            for WY in currStationWYs:
                currWY = currStation.sel(time=(currStation.water_year == WY))
                currT = currWY.T_avg
                currP = currWY.prcp         
                currTInterp = fillNANTemp(currT.values)
                currPInterp = fillNANPrcp(currP.values)            
                currTime = currWY.time
                currSWE = currWY.swe
                winterInds = np.logical_or(currTime.dt.month >= 11,currTime.dt.month <= 3)
                avgPrcpRate = np.nanmean(currPInterp[winterInds])
                numDaysBelowTThresh = np.sum(currTInterp<accumThresh)
                precipRates = np.append(precipRates,avgPrcpRate)
                numDaysBelowTThresh_s = np.append(numDaysBelowTThresh_s,numDaysBelowTThresh)
                obsPeakSWEs = np.append(obsPeakSWEs,currSWE.max().values)
                longestFreezeStreak = np.append(longestFreezeStreak,longestFreezingStreak(currTInterp))
                Tw = np.append(Tw,np.mean(currTInterp[winterInds]))
                
                # also include mean precip rate during snow covered season
                for i in range(len(currSWE)-10):
                    if np.sum(currSWE[i:i+10]!=0) == 10:
                        firstSnowInd = i
                        break
                dayOfFirstSnow = currTime.values[firstSnowInd]
                indMax = np.argmax(currSWE.values)
                dayOfMax = currTime.values[indMax]
                tMax_tStart_days = float(((dayOfMax-dayOfFirstSnow)/np.timedelta64(1, 's'))/86400)
                P_duringSnow = np.sum(currPInterp[firstSnowInd:indMax])/tMax_tStart_days
                precipRates_duringSnow = np.append(precipRates_duringSnow,P_duringSnow)
                prcpDuringTauF = np.append(prcpDuringTauF,currPInterp[currTInterp<=0].mean())
                ### Diagnostics ###
                # if numDaysBelowTThresh>300:
                    # print(currStation.name,stateName,WY)
            currStation = currStation.assign_attrs({'precipRates':precipRates,'numDaysBelowTThresh_s':numDaysBelowTThresh_s,
                                                    'peakSWE':obsPeakSWEs,'precipRates_duringSnow':precipRates_duringSnow,
                                                   'prcpDuringTauF':prcpDuringTauF,'longestFreezeStreak':longestFreezeStreak,'Tw':Tw})
            currStateDataNew = currStateDataNew + [currStation]
        allStateDictNew[stateName] = currStateDataNew
    return allStateDictNew  

# SWE Model

In [28]:
def sweModel(tempTS,precipTS,k,accumThresh=0):
    
    '''
    Simple model for the seasonal cycle of SWE.
    
    Inputs:
        - tempTS: timeseries of temperature over the course of a single water year; np.array, units of degree C
        - precipTS: timeseries of precipitation over the course of a single water year; np.array, units of mm
        - k: melt rate coefficient (units of mm/(degree day)); can be a single value or vary over the course of the water year
        - accumThresh: temperature threshold (in degree C) below which SWE accumulates (default is 0C)
    
    Outputs:
        - sweTS: timeseries of swe computed by the model (in units of mm)
    '''
    
    if type(k) != float and type(k) != int and len(k)!=2 and len(k)<300: # k is assumed to be a string and variable over the year
        k = timeVaryingK(k,len(tempTS))
    elif len(k) == 2: # k is a vector with paremeters mean and amp for flat sine
        argFact = 2*np.pi/365
        timeDomain = np.arange(len(tempTS))
        k = k[0]-k[1]*np.sin(timeDomain*argFact)
        k[k<.1] = .1
    elif len(k) >= 365: # k is an array of degree-day factor values for the whole year
        k = k
    else:
        k = np.ones(len(tempTS))*k
        
    accumTempMask = copy.deepcopy(tempTS)
    accumTempMask[tempTS<=accumThresh] = 1
    accumTempMask[tempTS>accumThresh] = 0
    accumTerm = precipTS*accumTempMask
    
    meltTerm = copy.deepcopy(tempTS)
    meltTerm[meltTerm<=0] = 0
    meltTerm[meltTerm>0] = meltTerm[meltTerm>0]*k[meltTerm>0]
    
    sweTS = np.array([])
    for iDay in range(len(tempTS)):
        if iDay == 0:
            newSWE = max(accumTerm[iDay]-meltTerm[iDay],0)
            sweTS = np.append(sweTS,newSWE)
        else:
            newSWE = max(sweTS[iDay-1] + accumTerm[iDay]-meltTerm[iDay],0)
            sweTS = np.append(sweTS,newSWE)
    
    return sweTS#,accumTerm,meltTerm

In [14]:
def sweModelVec(tempTS,precipTS,k,accumThresh=0):
    '''
    Simple model for the seasonal cycle of SWE. Same as the function sweModel, but vectorized for better performance. 
    
    Inputs:
        - tempTS: timeseries of temperature over the course of a single water year; np.array, units of degree C
        - precipTS: timeseries of precipitation over the course of a single water year; np.array, units of mm
        - k: melt rate coefficient (units of mm/(degree day)); can be a single value or vary over the course of the water year
        - accumThresh: temperature threshold (in degree C) below which SWE accumulates (default is 0C)
    
    Outputs:
        - sweTS: timeseries of swe computed by the model (in units of mm)
    '''
    
    if type(k) != float and type(k) != int and len(k)!=2 and len(k)<300: # k is assumed to be a string and variable over the year
        k = timeVaryingK(k,len(tempTS))
    elif len(k) == 2: # k is a vector with paremeters mean and amp for flat sine
        argFact = 2*np.pi/365
        timeDomain = np.arange(len(tempTS))
        k = k[0]-k[1]*np.sin(timeDomain*argFact)
        k[k<.1] = .1
    elif len(k) >= 365: # k is an array of degree-day factor values for the whole year
        k = k[:len(tempTS)]
    else:
        k = np.ones(len(tempTS))*k
        
    accumTempMask = copy.deepcopy(tempTS)
    accumTempMask[tempTS<=accumThresh] = 1
    accumTempMask[tempTS>accumThresh] = 0
    accumTerm = precipTS*accumTempMask
    
    meltTerm = copy.deepcopy(tempTS)
    meltTerm[meltTerm<=0] = 0
    meltTerm[meltTerm>0] = meltTerm[meltTerm>0]*k[meltTerm>0]
    
    
    sweTS = cumsum_breach_numba(accumTerm-meltTerm,0)
    
    
    #plt.plot(accumTerm-meltTerm)
    
    return sweTS

In [7]:
@njit
def cumsum_breach_numba(x,target):
    total = 0
    result = []
    for i, y in enumerate(x):
        total += y
        if total <= target:
            total = 0
            result.append(total)
        else:
            result.append(total)
    return result

In [18]:
def timeVaryingK(funcForm,lengthOfWY = 365, increaseDate=176, earlySeasonK=.5, meanOpt=np.nan, ampOpt=np.nan):
    '''
    Computes a time varying melt rate coefficient based on the provided funcForm.
    
    Inputs:
        - funcForm: string specifying the function used for k(t); options include: "logistic","sine","flatSine","flatSineOpt"
        - lengthOfWY: number of days in the water year
        - increaseDate: day (int) of water year where k increases at its fastest rate in the logistic formulation.
            Default is 176, which is ~March 25
        - earlySeasonK: minimum melt rate coefficient (mm/(degree day)) for logistic formulation
        - meanOpt: if funcForm=='flatSineOpt', supply the mean
        - ampOpt: if funcForm=='flatSineOpt', supply the amplitude
    Outputs:
        - k: melt rate coefficient that varies over the course of the water year
    '''
    
    if funcForm == 'logistic':
        timeDomain = np.arange(lengthOfWY)
        
        denom = 1+np.exp(-7.5e-2*(timeDomain-increaseDate))
        
        k = ((4-(1-earlySeasonK))/denom) + earlySeasonK
    
    elif funcForm == 'sine':
        timeDomain = np.arange(lengthOfWY)
        mean = 3
        amp = 2.5
        k = mean - amp*np.sin(timeDomain/365*2*np.pi+ 40/365*2*np.pi)
    
    elif funcForm == 'flatSine':
        argFact = 2*np.pi/365
        timeDomain = np.arange(lengthOfWY)
        mean = 1
        amp = 3
        k = mean-amp*np.sin(timeDomain*argFact)
        k[k<.1] = .1
    
    elif funcForm == 'flatSineOpt': # values calculated in snowpack/snotelAnalysis/optimization/optimizeSWEModel.py
        argFact = 2*np.pi/365
        timeDomain = np.arange(lengthOfWY)
        mean = meanOpt
        amp = ampOpt
        k = mean-amp*np.sin(timeDomain*argFact)
        k[k<.1] = .1
    return k

# Time Series processing

In [1]:
def calculateTauS(sweTimeSeries,minSWE=0):
    '''Return the length of the longest stretch with SWE>=minSWE in sweTimeSeries. sweTimeSeries
    is assumed to be 1 water year.
    '''
    tauSCountList = []

    currTauSCount = 0
    for swe in sweTimeSeries:
        if swe>minSWE:
            currTauSCount = currTauSCount + 1
        if swe<=minSWE:
            tauSCountList = tauSCountList + [currTauSCount]
            currTauSCount = 0

    return max(tauSCountList)

In [2]:
def numberOfFreezingDays(tempWY):
    '''
    Given a time series of daily mean temperature for a single year (tempWY), return the number of days with daily mean
    temperature below 0.
    '''
    return len(tempWY[tempWY<=0])

In [1]:
def longestFreezingStreak(timeSeries):
    '''Longest streak of consecutive below freezing days'''
    boolList = timeSeries<0
    return len(max([list(group) for item, group in groupby(boolList) if item==1], key=len))

In [1]:
## group data into moving bin and average ##

def binMeanFunction(dataToBin,xForData,binWidth,binIncrement):
    '''
    group data into moving bins of width binWidth and average
    '''

    if binIncrement>.1:
        startingPoint = round(np.min(xData))-1
        endingPoint = round(np.max(xData))+1
    else:
        startingPoint = np.min(xForData) - .1*abs(np.min(xForData))
        endingPoint = np.max(xForData)+.1*abs(np.max(xForData))
    binStart = startingPoint
    binEnd = startingPoint+binWidth

    binnedMean = np.array([])
    binMidPt = np.array([])
    while binEnd<endingPoint:
        indSelect = np.logical_and(xForData>binStart,xForData<binEnd)
        binnedMean = np.append(binnedMean,np.mean(dataToBin[indSelect]))
        binMidPt = np.append(binMidPt,(binStart+binEnd)/2)
        binStart = binStart + binIncrement
        binEnd = binEnd + binIncrement
    return binMidPt,binnedMean

# Cosine Seasonal Cycle Functions

In [25]:
def Tw_fromCos(annualMeanTemp,seasonalCycleAmp):
    '''
    Calculate winter-mean temp given an annual-mean 
    temperature and a seasonal cycle amplitude.
    '''
    try:
        Tw = np.array([])
        for i in range(len(annualMeanTemp)):
            omega = 2*np.pi/365
            timeDomainWinter = np.arange(109,260)
            T_timeSeriesWinter = annualMeanTemp[i] + seasonalCycleAmp*np.cos(omega*timeDomainWinter)
            Tw_i = np.trapz(T_timeSeriesWinter)/(260-109)
            Tw = np.append(Tw,Tw_i)
        return Tw
    except TypeError:
        omega = 2*np.pi/365
        timeDomainWinter = np.arange(109,260)
        T_timeSeriesWinter = annualMeanTemp + seasonalCycleAmp*np.cos(omega*timeDomainWinter)
        Tw_i = np.trapz(T_timeSeriesWinter)/(260-109)
        return Tw_i

def annualMean_fromTw(Tw,seasonalCycleAmp):
    '''
    Calculate annual-mean temp given a winter-mean 
    temperature and a seasonal cycle amplitude.
    '''
    try:
        annualMeanTemp = np.array([])
        for i in range(len(Tw)):
            omega = 2*np.pi/365
            timeDomainWinter = np.arange(109,260)
            T_prime = seasonalCycleAmp*np.cos(omega*timeDomainWinter)
            annualMeanTemp_i = Tw[i] - (1/(260-109))*np.trapz(T_prime)
            annualMeanTemp = np.append(annualMeanTemp,annualMeanTemp_i)
        return annualMeanTemp
    except TypeError:
        omega = 2*np.pi/365
        timeDomainWinter = np.arange(109,260)
        T_prime = seasonalCycleAmp*np.cos(omega*timeDomainWinter)
        annualMeanTemp_i = Tw - (1/(260-109))*np.trapz(T_prime)
        return annualMeanTemp_i

In [9]:
def E_tauF_normalDistDaily(amp,s,annualMeanTempRange=np.arange(-30,31,1)):
    '''
    Expected value of \tau_F when assuming a cosine temperature cycle with Gaussian daily-mean temperature noise with standard deviation s.
    '''
    omega = 2*np.pi/365
    time = np.arange(0,365)
    
    E_tauFs_theory = np.array([])

    for T in (annualMeanTempRange):
            cosineTemp = T-amp*np.cos(omega*time)
            if s!=0:
                expTauFTheory = np.sum(scipy.stats.norm(loc=cosineTemp,scale=s).cdf(0))
            elif s==0:
                expTauFTheory = numberOfFreezingDays(cosineTemp)
            E_tauFs_theory = np.append(E_tauFs_theory,expTauFTheory)
        
    return E_tauFs_theory

# Data Analysis Tools

In [18]:
def R2_from1to1(scatterPointsX,scatterPointsY):
    '''
    Calculates the R^2 value for the points given in scatterPointsX and scatterPointsY.
    
    Inputs:
        - scatterPointsX: array of x coordinates
        - scatterPointsY: array of y coordinates (this is the variable whose variance you are trying to explain).
    
    Outputs: 
        - R2: R^2 value
    '''
    f = scatterPointsX # fitted values are the same as scatterPointsX
    residuals = scatterPointsY-f
    SS_res = np.sum(residuals**2)
    
    meanY = np.mean(scatterPointsY)
    SS_tot = np.sum((scatterPointsY-meanY)**2)
    
    R2 = 1 - (SS_res/SS_tot)
    
    return R2

In [7]:
def zeroLowerBound(x,on=True):
    '''
    Returns 0 if x<0 and x otherwise
    
    Inputs:
        - x: number to be tested
        - on: option to turn the function on and off
    Returns:
        - updated x (based on whether the function is on or off)
    '''
    if on:
        try:
            if len(x)>0: # here x should be an np.array
                x = copy.deepcopy(x)
                x[x<0] = 0
                return x
        except TypeError:
            if x>0:
                return x
            else:
                return 0
    else:
        return x

In [2]:
def leastSquaresSlope(valsX,valsY):
    '''
    Determine the slope between a scatter plot of valsX and valsY and minimizes the 
    square of the residual. Also calculate the R^2 value associated with the 
    optimum point.
    '''
    
    model = lambda beta: valsX*beta
    minSumRes = lambda beta: np.nansum((valsY-model(beta))**2)

    beta_opt = minimize(minSumRes,1,method = 'Nelder-Mead').x[0]
    
    residuals = valsY-model(beta_opt)
    SSres = np.nansum(residuals**2)
    SStot = np.nansum((valsY-np.nanmean(valsY))**2)
    R2 = 1-(SSres/SStot)
    
    return beta_opt,R2

In [9]:
def covariance(x,y):
    '''
    Simply compute the covariance of x and y following the definition of covariance.
    '''
    fact1 = x-np.mean(x)
    fact2 = y-np.mean(y)
    cov = np.mean(fact1*fact2)
    return cov

In [ ]:
def movingAverage(a, N):
    '''
    Returns moving average of a (evenly weighted) over N data points. 
    The output is the same length as a.
    '''
    y_padded = np.pad(a, (N//2, N-1-N//2), mode='edge')
    y_smooth = np.convolve(y_padded, np.ones((N,))/N, mode='valid')
    return y_smooth

In [2]:
# =========================================================
# density_estimation.py
# ---------------------------------------------------------
# Estimating densities from data using a diffusion-based method.
# =========================================================
# functions -----------------------------------------------
def kde1d(x, n=1024, limits=None):
    """
    Estimates the 1d density from discrete observations.

    The input is a list/array `x` of numbers that represent discrete
    observations of a random variable. They are binned on a grid of
    `n` points within the data `limits`, if specified, or within
    the limits given by the values' range. `n` will be coerced to the
    next highest power of two if it isn't one to begin with.

    The limits may be given as a tuple (`xmin`, `xmax`) or a single
    number denoting the upper bound of a range centered at zero.
    If any of those values are `None`, they will be inferred from the
    data.

    After binning, the function determines the optimal bandwidth
    according to the diffusion-based method. It then smooths the
    binned data over the grid using a Gaussian kernel with a standard
    deviation corresponding to that bandwidth.

    Returns the estimated `density` and the `grid` upon which it was
    computed, as well as the optimal `bandwidth` value the algorithm
    determined. Raises `ValueError` if the algorithm did not converge.
    """
    # Load packages.
    from numpy import array, arange
    from numpy import exp, sqrt, pi as π
    from numpy import ceil, log2
    from numpy import prod as product
    from numpy import histogram
    from scipy.fft import dct, idct
    from scipy.optimize import brentq

    # Convert to array in case a list is passed in.
    x = array(x)

    # Round up number of bins to next power of two.
    n = int(2**ceil(log2(n)))

    # Determine missing data limits.
    if limits is None:
        xmin = xmax = None
    elif isinstance(limits, tuple):
        (xmin, xmax) = limits
    else:
        xmin = -limits
        xmax = +limits
    if None in (xmin, xmax):
        delta = x.max() - x.min()
        if xmin is None:
            xmin = x.min() - delta/10
        if xmax is None:
            xmax = x.max() + delta/10

    # Determine data range, required for scaling.
    Δx = xmax - xmin

    # Determine number of data points.
    N = len(x)

    # Bin samples on regular grid.
    (binned, edges) = histogram(x, bins=n, range=(xmin, xmax))
    grid = edges[:-1]

    # Compute 2d discrete cosine transform, then adjust first component.
    transformed = dct(binned/N)
    transformed[0] /= 2

    # Pre-compute squared indices and transform components before solver loop.
    k  = arange(n, dtype='float')      # "float" avoids integer overflow.
    k2 = k**2
    a2 = (transformed/2)**2

    # Define internal function to be solved iteratively.
    def ξγ(t, l=7):
        """Returns ξ γ^[l] as a function of diffusion time t."""
        f = 2*π**(2*l) * sum(k2**l * a2 * exp(-π**2 * k2*t))
        for s in range(l-1, 1, -1):
            K = product(range(1, 2*s, 2)) / sqrt(2*π)
            C = (1 + (1/2)**(s+1/2)) / 3
            t = (2*C*K/N/f)**(2/(3+2*s))
            f = 2*π**(2*s) * sum(k2**s * a2 * exp(-π**2 * k2*t))
        return (2*N*sqrt(π)*f)**(-2/5)

    # Solve for optimal diffusion time t*.
    try:
        ts = brentq(lambda t: t - ξγ(t), 0, 0.1)
    except ValueError:
        raise ValueError('Bandwidth optimization did not converge.') from None

    # Apply Gaussian filter with optimized kernel.
    smoothed = transformed * exp(-π**2 * ts/2 * k**2)

    # Reverse transformation after adjusting first component.
    smoothed[0] *= 2
    inverse = idct(smoothed)

    # Normalize density.
    density = inverse * n/Δx

    # Determine bandwidth from diffusion time.
    bandwidth = sqrt(ts) * Δx

    # Make output array.
    pdf = array((grid, density)).T

    # Return results.
    return pdf

In [1]:
def flatten(xss):
    '''
    For flattening lists
    '''
    return [x for xs in xss for x in xs]

# Diagnostics

In [10]:
def checkIntPrcpTotPrcp(stateNames,allStateDict,upToDate=None,stationBiasBoolDefault=True):
    '''
    Check that integrated precipitation equals the accumulated precipitation variable.
    
    Inputs:
        - stateNames: list of state names (the keys in allStateDict)
        - allStateDict: station data for the stations in the states allStateDict
        - upToDate: (optional,int) month that will compare integrated precip and the accumulated precip variable up to the 15th of the specified month. Must be less than 10.
        - stationBiasBoolDefault: boolean. Decides whether stations with no indication of temperature bias correction should
            be included. Default is True.
    
    Outputs:
        prints stations and WY where the summed precip does not equal the accumulated precip variable
    '''
    
    for iState,stateName in enumerate(stateNames):
        if iState == 0:
            print(stateName)
        else:
            print('\n'+stateName)
            
        currStateData = allStateDict[stateName]
        currStateDataNew = [] # list of modified stations

        for iS,currStation in (enumerate(currStateData)):
            print(f'{iS}/{len(currStateData)}', end="\r")

            # only look at stations with corrected temp
            try:
                stationBiasBool = stationBiasDict[currStation.number]=='Complete'
            except KeyError:
                stationBiasBool = stationBiasBoolDefault
                #print('No bias data')

            if stationBiasBool:

                currStationWYs = np.unique(currStation.water_year.values)

                peakSWEObs = np.array([])
                peakSWEModel = np.array([])

                for WY in currStationWYs:
                    currWY = currStation.sel(time=(currStation.water_year == WY))

                    # only include years with sufficient data
                    check1 = checkForEnoughData(currWY.swe,fractionFull=minDataPercent)
                    check2 = checkForEnoughData(currWY.prcp,fractionFull=minDataPercent)
                    check3 = checkForEnoughData(currWY.T_avg,fractionFull=minDataPercent)
                    if check1 and check2 and check3:
                        
                        if type(upToDate)==int:
                            currWYTime = currWY.time.values
                            if upToDate > 9:
                                print('upToDate must be less than 10')
                                return
                            timeInd = np.nonzero(currWYTime==np.datetime64(f'{WY}-0{upToDate}-15'))[0][0]
                            currP = currWY.prcp.cumsum()[:timeInd].max().values
                            currPTot = currWY.prcpTot[:timeInd].max().values
                        else:
                            currP = currWY.prcp.cumsum().max().values
                            currPTot = currWY.prcpTot.max().values
                        
                        if abs(currP-currPTot)/currPTot > .05:
                            currPTot = currWY.prcpTot[:timeInd+1].max().values
                            if abs(currP-currPTot)/currPTot > .05:
                                print(f'{currStation.name}; {WY}')
                                print(currPTot)
                                print(currP)
                            

# Plotting

In [2]:
def shiftedColorMap(cmap, start=0, midpoint=0.5, stop=1.0, name='shiftedcmap'):
    '''
    <<< Function from https://stackoverflow.com/questions/7404116/defining-the-midpoint-of-a-colormap-in-matplotlib >>>
    
    Function to offset the "center" of a colormap. Useful for
    data with a negative min and positive max and you want the
    middle of the colormap's dynamic range to be at zero.

    Input
    -----
      cmap : The matplotlib colormap to be altered
      start : Offset from lowest point in the colormap's range.
          Defaults to 0.0 (no lower offset). Should be between
          0.0 and `midpoint`.
      midpoint : The new center of the colormap. Defaults to 
          0.5 (no shift). Should be between 0.0 and 1.0. In
          general, this should be  1 - vmax / (vmax + abs(vmin))
          For example if your data range from -15.0 to +5.0 and
          you want the center of the colormap at 0.0, `midpoint`
          should be set to  1 - 5/(5 + 15)) or 0.75
      stop : Offset from highest point in the colormap's range.
          Defaults to 1.0 (no upper offset). Should be between
          `midpoint` and 1.0.
    '''
    cdict = {
        'red': [],
        'green': [],
        'blue': [],
        'alpha': []
    }

    # regular index to compute the colors
    reg_index = np.linspace(start, stop, 257)

    # shifted index to match the data
    shift_index = np.hstack([
        np.linspace(0.0, midpoint, 128, endpoint=False), 
        np.linspace(midpoint, 1.0, 129, endpoint=True)
    ])

    for ri, si in zip(reg_index, shift_index):
        r, g, b, a = cmap(ri)

        cdict['red'].append((si, r, r))
        cdict['green'].append((si, g, g))
        cdict['blue'].append((si, b, b))
        cdict['alpha'].append((si, a, a))

    newcmap = matplotlib.colors.LinearSegmentedColormap(name, cdict)
    #matplotlib.colormaps.register(newcmap)

    return newcmap


In [1]:
# from https://matplotlib.org/stable/gallery/text_labels_and_annotations/angle_annotation.html

from matplotlib.patches import Arc
from matplotlib.transforms import Bbox, IdentityTransform, TransformedBbox


class AngleAnnotation(Arc):
    """
    Draws an arc between two vectors which appears circular in display space.
    """
    def __init__(self, xy, p1, p2, size=75, unit="points", ax=None,
                 text="", textposition="inside", text_kw=None, **kwargs):
        """
        Parameters
        ----------
        xy, p1, p2 : tuple or array of two floats
            Center position and two points. Angle annotation is drawn between
            the two vectors connecting *p1* and *p2* with *xy*, respectively.
            Units are data coordinates.

        size : float
            Diameter of the angle annotation in units specified by *unit*.

        unit : str
            One of the following strings to specify the unit of *size*:

            * "pixels": pixels
            * "points": points, use points instead of pixels to not have a
              dependence on the DPI
            * "axes width", "axes height": relative units of Axes width, height
            * "axes min", "axes max": minimum or maximum of relative Axes
              width, height

        ax : `matplotlib.axes.Axes`
            The Axes to add the angle annotation to.

        text : str
            The text to mark the angle with.

        textposition : {"inside", "outside", "edge"}
            Whether to show the text in- or outside the arc. "edge" can be used
            for custom positions anchored at the arc's edge.

        text_kw : dict
            Dictionary of arguments passed to the Annotation.

        **kwargs
            Further parameters are passed to `matplotlib.patches.Arc`. Use this
            to specify, color, linewidth etc. of the arc.

        """
        self.ax = ax or plt.gca()
        self._xydata = xy  # in data coordinates
        self.vec1 = p1
        self.vec2 = p2
        self.size = size
        self.unit = unit
        self.textposition = textposition

        super().__init__(self._xydata, size, size, angle=0.0,
                         theta1=self.theta1, theta2=self.theta2, **kwargs)

        self.set_transform(IdentityTransform())
        self.ax.add_patch(self)

        self.kw = dict(ha="center", va="center",
                       xycoords=IdentityTransform(),
                       xytext=(0, 0), textcoords="offset points",
                       annotation_clip=True)
        self.kw.update(text_kw or {})
        self.text = ax.annotate(text, xy=self._center, **self.kw)

    def get_size(self):
        factor = 1.
        if self.unit == "points":
            factor = self.ax.figure.dpi / 72.
        elif self.unit[:4] == "axes":
            b = TransformedBbox(Bbox.unit(), self.ax.transAxes)
            dic = {"max": max(b.width, b.height),
                   "min": min(b.width, b.height),
                   "width": b.width, "height": b.height}
            factor = dic[self.unit[5:]]
        return self.size * factor

    def set_size(self, size):
        self.size = size

    def get_center_in_pixels(self):
        """return center in pixels"""
        return self.ax.transData.transform(self._xydata)

    def set_center(self, xy):
        """set center in data coordinates"""
        self._xydata = xy

    def get_theta(self, vec):
        vec_in_pixels = self.ax.transData.transform(vec) - self._center
        return np.rad2deg(np.arctan2(vec_in_pixels[1], vec_in_pixels[0]))

    def get_theta1(self):
        return self.get_theta(self.vec1)

    def get_theta2(self):
        return self.get_theta(self.vec2)

    def set_theta(self, angle):
        pass

    # Redefine attributes of the Arc to always give values in pixel space
    _center = property(get_center_in_pixels, set_center)
    theta1 = property(get_theta1, set_theta)
    theta2 = property(get_theta2, set_theta)
    width = property(get_size, set_size)
    height = property(get_size, set_size)

    # The following two methods are needed to update the text position.
    def draw(self, renderer):
        self.update_text()
        super().draw(renderer)

    def update_text(self):
        c = self._center
        s = self.get_size()
        angle_span = (self.theta2 - self.theta1) % 360
        angle = np.deg2rad(self.theta1 + angle_span / 2)
        r = s / 2
        if self.textposition == "inside":
            r = s / np.interp(angle_span, [60, 90, 135, 180],
                                          [3.3, 3.5, 3.8, 4])
        self.text.xy = c + r * np.array([np.cos(angle), np.sin(angle)])
        if self.textposition == "outside":
            def R90(a, r, w, h):
                if a < np.arctan(h/2/(r+w/2)):
                    return np.sqrt((r+w/2)**2 + (np.tan(a)*(r+w/2))**2)
                else:
                    c = np.sqrt((w/2)**2+(h/2)**2)
                    T = np.arcsin(c * np.cos(np.pi/2 - a + np.arcsin(h/2/c))/r)
                    xy = r * np.array([np.cos(a + T), np.sin(a + T)])
                    xy += np.array([w/2, h/2])
                    return np.sqrt(np.sum(xy**2))

            def R(a, r, w, h):
                aa = (a % (np.pi/4))*((a % (np.pi/2)) <= np.pi/4) + \
                     (np.pi/4 - (a % (np.pi/4)))*((a % (np.pi/2)) >= np.pi/4)
                return R90(aa, r, *[w, h][::int(np.sign(np.cos(2*a)))])

            bbox = self.text.get_window_extent()
            X = R(angle, r, bbox.width, bbox.height)
            trans = self.ax.figure.dpi_scale_trans.inverted()
            offs = trans.transform(((X-s/2), 0))[0] * 72
            self.text.set_position([offs*np.cos(angle), offs*np.sin(angle)])
